In [1]:
import os
import sys
import json
import glob
import gc

import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType

In [2]:
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

In [17]:
#Saving 2026 prediction to snowflake 
parquet_pred = pd.read_parquet(r"c:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#2_calendar_attributes\Modelling\predictions_2026_negbin_torch_convention\20260918_160112_3f294f6d\daily_tft_negbin_scooters_2026-09-15_20_02_52_predictions.parquet")

parquet_pred_sf = session.create_dataframe(parquet_pred)

parquet_pred_sf.write.mode('overwrite').save_as_table('MOP_DATABASE.SOQ.DAILY_PREDICTIONS_DEALER_SKU_FAMILY_LEVEL_ITERATION2')

In [4]:
filtered_df=parquet_pred_sf.filter((F.col("CAL_DATE")>='2026-09-01') & (F.col("CAL_DATE")<='2026-09-30'))

filtered_df.select(F.sum("PRED_Q70").alias("TOTAL_SALES_FOR_PRED_Q70"),F.sum("PRED_Q75").alias("TOTAL_SALES_FOR_PRED_Q75")).show()

-----------------------------------------------------------
|"TOTAL_SALES_FOR_PRED_Q70"  |"TOTAL_SALES_FOR_PRED_Q75"  |
-----------------------------------------------------------
|377824.0                    |453379.0                    |
-----------------------------------------------------------



In [5]:
parquet_pred_sf.filter((F.col("CAL_DATE")>='2026-09-26') & (F.col("CAL_DATE")<='2026-09-30')).group_by(F.col("CAL_DATE")).agg(F.sum("PRED_Q70")).show()

-----------------------------------------
|"CAL_DATE"           |"SUM(PRED_Q70)"  |
-----------------------------------------
|2026-09-26 00:00:00  |1513.0           |
|2026-09-28 00:00:00  |8519.0           |
|2026-09-29 00:00:00  |8223.0           |
|2026-09-30 00:00:00  |7543.0           |
|2026-09-27 00:00:00  |1744.0           |
-----------------------------------------



In [6]:
parquet_pred.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,CAL_DATE,PRED_MEAN,PRED_Q45,PRED_Q55,PRED_Q70,PRED_Q75
0,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-01,0.258082,0.0,0.0,0.0,0.0
1,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-02,0.309746,0.0,0.0,0.0,0.0
2,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-03,0.303961,0.0,0.0,0.0,0.0
3,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-04,0.415599,0.0,0.0,1.0,1.0
4,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-05,0.276236,0.0,0.0,0.0,0.0


In [7]:
#September predictions 
sep_data = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#2_calendar_attributes\Modelling\Final output\September_predictions_q70_iteration2.csv",parse_dates=True,date_format='%Y-%m=%d')

In [8]:
sep_data.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,CAL_DATE,PRED_Q70,MONTH_NAME
0,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-01,0.0,September
1,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-02,0.0,September
2,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-03,0.0,September
3,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-04,1.0,September
4,11132_XTREME 125_DRUM_SELF_CAST_GREY,2026-09-05,0.0,September


In [9]:
sep_data.loc[sep_data["CAL_DATE"]>='2026-09-26',:].groupby("CAL_DATE").agg(daily_sales=("PRED_Q70","sum"))

,daily_sales
CAL_DATE,
2026-09-26,1513.0
2026-09-27,1744.0
2026-09-28,8519.0
2026-09-29,8223.0
2026-09-30,7543.0


In [10]:
df = session.sql("""
    SELECT CAL_DATE,NET_SALES
    FROM MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES
    WHERE PARENT_DEALER_CODE_MODEL_FAMILY IN (
                 SELECT *
                 FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)
    AND CAL_DATE BETWEEN '2023-09-14' AND '2023-12-12'
    UNION ALL
    SELECT CAL_DATE,NET_SALES
    FROM MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES
    WHERE CAL_DATE BETWEEN '2024-09-02' AND '2024-11-30'
    AND PARENT_DEALER_CODE_MODEL_FAMILY IN (
                 SELECT *
                 FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)
    UNION ALL
    SELECT CAL_DATE,NET_SALES
    FROM MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES
    WHERE CAL_DATE BETWEEN '2025-08-23' AND '2025-11-19'
        AND PARENT_DEALER_CODE_MODEL_FAMILY IN (
                 SELECT *
                 FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)
    UNION ALL
    SELECT CAL_DATE,PRED_Q70 AS NET_SALES
    FROM MOP_DATABASE.SOQ.DAILY_PREDICTIONS_DEALER_SKU_FAMILY_LEVEL_ITERATION2
    WHERE CAL_DATE>='2026-09-11'
    """)

In [11]:
data = df.group_by("CAL_DATE").agg(F.sum("NET_SALES").alias("TOTAL_SALES")).sort(F.col("CAL_DATE").asc())
data.select(F.max("CAL_DATE").alias("MAXIMUM_CALENDAR_DATE")).show()

---------------------------
|"MAXIMUM_CALENDAR_DATE"  |
---------------------------
|2026-12-07 00:00:00      |
---------------------------



In [12]:
data.filter(((F.col("CAL_DATE")>='2026-09-26') & (F.col("CAL_DATE")<='2026-09-30'))).show()

---------------------------------------
|"CAL_DATE"           |"TOTAL_SALES"  |
---------------------------------------
|2026-09-26 00:00:00  |1513.0         |
|2026-09-27 00:00:00  |1744.0         |
|2026-09-28 00:00:00  |8519.0         |
|2026-09-29 00:00:00  |8223.0         |
|2026-09-30 00:00:00  |7543.0         |
---------------------------------------



In [13]:
case_expression = """CASE 
        WHEN CAL_DATE<'2023-09-29' THEN 'PP_MINUS_'||DATEDIFF(DAY,CAL_DATE,'2023-09-29'::DATE)
        WHEN CAL_DATE BETWEEN '2023-09-29' AND '2023-10-14' THEN 'PP'||(DATEDIFF(DAY,'2023-09-29'::DATE,CAL_DATE)+1)
        WHEN CAL_DATE BETWEEN '2023-10-15' AND '2023-10-24' THEN 'NAVRATARI'||(DATEDIFF(DAY,'2023-10-15'::DATE,CAL_DATE)+1)
        WHEN CAL_DATE BETWEEN '2023-10-25' AND '2023-11-11' THEN 'D_MINUS_'||DATEDIFF(DAY,CAL_DATE,'2023-11-12'::DATE)
        WHEN CAL_DATE = '2023-11-12' THEN 'DIWALI'
        WHEN CAL_DATE BETWEEN '2023-11-13' AND '2023-12-12' THEN 'D_PLUS_'||DATEDIFF(DAY,'2023-11-12'::DATE,CAL_DATE)
        
        WHEN CAL_DATE < '2024-09-17' THEN 'PP_MINUS_'||DATEDIFF(DAY, CAL_DATE, '2024-09-17'::DATE)
        WHEN CAL_DATE BETWEEN '2024-09-17' AND '2024-10-02' THEN 'PP'||(DATEDIFF(DAY, '2024-09-17'::DATE, CAL_DATE) + 1)
        WHEN CAL_DATE BETWEEN '2024-10-03' AND '2024-10-12' THEN 'NAVRATARI'||(DATEDIFF(DAY, '2024-10-03'::DATE, CAL_DATE) + 1)
        WHEN CAL_DATE BETWEEN '2024-10-13' AND '2024-10-30' THEN 'D_MINUS_'||DATEDIFF(DAY, CAL_DATE, '2024-10-31'::DATE)
        WHEN CAL_DATE = '2024-10-31' THEN 'DIWALI'
        WHEN CAL_DATE BETWEEN '2024-11-01' AND '2024-11-30' THEN 'D_PLUS_'||DATEDIFF(DAY, '2024-10-31'::DATE, CAL_DATE)


        WHEN CAL_DATE < '2025-09-07' THEN 'PP_MINUS_'||DATEDIFF(DAY, CAL_DATE, '2025-09-07'::DATE)
        WHEN CAL_DATE BETWEEN '2025-09-07' AND '2025-09-21' THEN 'PP'||(DATEDIFF(DAY, '2025-09-07'::DATE, CAL_DATE) + 1)
        WHEN CAL_DATE BETWEEN '2025-09-22' AND '2025-10-01' THEN 'NAVRATARI'||(DATEDIFF(DAY, '2025-09-22'::DATE, CAL_DATE) + 1)
        WHEN CAL_DATE BETWEEN '2025-10-02' AND '2025-10-19' THEN 'D_MINUS_'||DATEDIFF(DAY, CAL_DATE, '2025-10-20'::DATE)
        WHEN CAL_DATE = '2025-10-20' THEN 'DIWALI'
        WHEN CAL_DATE BETWEEN '2025-10-21' AND '2025-11-19' THEN 'D_PLUS_'||DATEDIFF(DAY, '2025-10-20'::DATE, CAL_DATE)

        WHEN CAL_DATE < '2026-09-26' THEN 'PP_MINUS_'||DATEDIFF(DAY, CAL_DATE, '2026-09-26'::DATE)
        WHEN CAL_DATE BETWEEN '2026-09-26' AND '2026-10-10' THEN 'PP'||(DATEDIFF(DAY, '2026-09-26'::DATE, CAL_DATE) + 1)
        WHEN CAL_DATE BETWEEN '2026-10-11' AND '2026-10-20' THEN 'NAVRATARI'||(DATEDIFF(DAY, '2026-10-11'::DATE, CAL_DATE) + 1)
        WHEN CAL_DATE BETWEEN '2026-10-21' AND '2026-11-07' THEN 'D_MINUS_'||DATEDIFF(DAY, CAL_DATE, '2026-11-08'::DATE)
        WHEN CAL_DATE = '2026-11-08' THEN 'DIWALI'
        WHEN CAL_DATE BETWEEN '2026-11-09' AND '2026-12-08' THEN 'D_PLUS_'||DATEDIFF(DAY, '2026-11-08'::DATE, CAL_DATE)

END"""

data = data.with_column("TYPE_OF_DAY",F.expr(case_expression))


In [14]:
df = data.to_pandas()

In [15]:
df.loc[((df["CAL_DATE"]>='2026-09-26') & (df["CAL_DATE"]<='2026-09-30')),:]

,CAL_DATE,TOTAL_SALES,TYPE_OF_DAY
284,2026-09-26,1513.0,PP1
285,2026-09-27,1744.0,PP2
286,2026-09-28,8519.0,PP3
287,2026-09-29,8223.0,PP4
288,2026-09-30,7543.0,PP5


In [16]:
import pandas as pd
import plotly.express as px

# 1. Ensure CAL_DATE is a proper datetime object
df["CAL_DATE"] = pd.to_datetime(df["CAL_DATE"])

# 2. Create a YEAR column for the legend (convert to string so Plotly treats it as discrete categories)
df["YEAR"] = df["CAL_DATE"].dt.year.astype(str)

# 3. Filter for the specific years you want (Optional, if your df has more years)
# df_filtered = df[df["YEAR"].isin(["2023", "2024", "2025","2026"])]

# 4. Extract the exact chronological order of TYPE_OF_DAY based on CAL_DATE
# Sorting by date and dropping duplicates preserves the true timeline sequence
chronological_order = df.sort_values("CAL_DATE")["TYPE_OF_DAY"].drop_duplicates().tolist()


# 5. Create the line chart
fig = px.line(
    df,
    x="TYPE_OF_DAY",
    y="TOTAL_SALES",
    color="YEAR",
    title="Total Sales Over Festive Period (2023-2026) : Series A comparison",
    markers=True,
    labels={"TYPE_OF_DAY": "Day of Festive Period", "TOTAL_SALES": "Total Sales"}
)

# 6. Force the X-axis to follow our extracted chronological order
fig.update_xaxes(categoryorder="array", categoryarray=chronological_order)

fig.write_html("festive_days.html")
# session.file.put("file:///tmp/festive_days.html", "@MOP_DATABASE.SOQ.TFT_DAILY_FORECASTING_FILES", auto_compress=False, overwrite=True)

# Display the chart
fig.show()

In [17]:
navratari_days = [i for i in df["TYPE_OF_DAY"].unique().tolist() if 'NAVRATARI' in i]

In [18]:
#NAVRATARI 

navratari_df = df.loc[df["TYPE_OF_DAY"].isin(navratari_days),:]

navratari_df=navratari_df.groupby("YEAR",as_index=False).agg(AVG_NAVRATARI_SALES=("TOTAL_SALES","mean"))
navratari_df

,YEAR,AVG_NAVRATARI_SALES
0,2023,32199.7
1,2024,36434.4
2,2025,31604.6
3,2026,44915.3


In [19]:
#PP_days 

pp_days = [i for i in df["TYPE_OF_DAY"].unique().tolist() if (('PP' in i) and ('PP_' not in i))]

pp_df = df.loc[df["TYPE_OF_DAY"].isin(pp_days),:]
pp_df=pp_df.groupby("YEAR",as_index=False).agg(AVG_PP_DAYS_SALES=("TOTAL_SALES","mean"))
pp_df

,YEAR,AVG_PP_DAYS_SALES
0,2023,3721.875000
1,2024,3394.562500
2,2025,2810.600000
3,2026,2733.933333


In [20]:
#Pre_pp_days 

pre_pp_days = [i for i in df["TYPE_OF_DAY"].unique().tolist() if (('PP_' in i))]

pre_pp_df = df.loc[df["TYPE_OF_DAY"].isin(pre_pp_days),:]
pre_pp_df=pre_pp_df.groupby("YEAR",as_index=False).agg(AVG_PRE_PP_DAYS_SALES=("TOTAL_SALES","mean"))
pre_pp_df

,YEAR,AVG_PRE_PP_DAYS_SALES
0,2023,9552.133333
1,2024,7741.933333
2,2025,9295.200000
3,2026,14606.400000


In [21]:
#Navratari start 

first_day_of_navratari_df=df.loc[df["TYPE_OF_DAY"].isin(['NAVRATARI1']),["YEAR","TOTAL_SALES"]].drop_duplicates().rename(columns={"TOTAL_SALES":"NAVRATARI_FIRST_DAY_SALES"})
first_day_of_navratari_df.reset_index(drop=True,inplace=True)
first_day_of_navratari_df

,YEAR,NAVRATARI_FIRST_DAY_SALES
0,2023,57230.0
1,2024,71797.0
2,2025,46996.0
3,2026,64462.0


In [22]:
last_day_of_navratari_df=df.loc[df["TYPE_OF_DAY"].isin(['NAVRATARI9']),["YEAR","TOTAL_SALES"]].drop_duplicates().rename(columns={"TOTAL_SALES":"NAVRATARI_LAST_DAY_SALES"})
last_day_of_navratari_df.reset_index(drop=True,inplace=True)
last_day_of_navratari_df

,YEAR,NAVRATARI_LAST_DAY_SALES
0,2023,34530.0
1,2024,38061.0
2,2025,32365.0
3,2026,59173.0


In [23]:
dussehra_df=df.loc[df["TYPE_OF_DAY"].isin(['NAVRATARI10']),["YEAR","TOTAL_SALES"]].drop_duplicates().rename(columns={"TOTAL_SALES":"DUSSEHRA_SALES"})
dussehra_df.reset_index(drop=True,inplace=True)
dussehra_df

,YEAR,DUSSEHRA_SALES
0,2023,54390.0
1,2024,59639.0
2,2025,33493.0
3,2026,29502.0


In [24]:
#Pre_diwali_days 

pre_diwali_days = [i for i in df["TYPE_OF_DAY"].unique() if ('D_MINUS' in i)]

pre_diwali_df = df.loc[df["TYPE_OF_DAY"].isin(pre_diwali_days),:]
pre_diwali_df=pre_diwali_df.groupby("YEAR",as_index=False).agg(AVG_PRE_DIWALI_DAYS_SALES=("TOTAL_SALES","mean"))
pre_diwali_df

,YEAR,AVG_PRE_DIWALI_DAYS_SALES
0,2023,32225.611111
1,2024,36351.222222
2,2025,42761.444444
3,2026,40313.666667


In [25]:
#Diwali
diwali_df=df.loc[df["TYPE_OF_DAY"]=='DIWALI',["YEAR","TOTAL_SALES"]].drop_duplicates().rename(columns={"TOTAL_SALES":"DIWALI_SALES"})
diwali_df.reset_index(drop=True,inplace=True)
diwali_df

,YEAR,DIWALI_SALES
0,2023,85119.0
1,2024,85507.0
2,2025,100113.0
3,2026,110043.0


In [26]:
#Dhanteras
dhanteras_df=df.loc[df["TYPE_OF_DAY"]=='D_MINUS_2',["YEAR","TOTAL_SALES"]].drop_duplicates().rename(columns={"TOTAL_SALES":"DHANTERAS_SALES"})
dhanteras_df.reset_index(drop=True,inplace=True)
dhanteras_df

,YEAR,DHANTERAS_SALES
0,2023,248794.0
1,2024,293987.0
2,2025,245243.0
3,2026,242579.0


In [27]:
#Post Diwali

post_diwali_days = [i for i in df["TYPE_OF_DAY"].unique() if ('D_PLUS' in i)]
post_diwali_df = df.loc[df["TYPE_OF_DAY"].isin(post_diwali_days),:]
post_diwali_df=post_diwali_df.groupby("YEAR",as_index=False).agg(AVG_POST_DIWALI_DAYS_SALE=("TOTAL_SALES","mean"))
post_diwali_df

,YEAR,AVG_POST_DIWALI_DAYS_SALE
0,2023,16409.133333
1,2024,19085.066667
2,2025,21435.700000
3,2026,25637.241379


In [29]:
from functools import reduce

list_of_dfs = [pre_pp_df, pp_df, first_day_of_navratari_df, navratari_df,
               last_day_of_navratari_df, dussehra_df, pre_diwali_df,
               dhanteras_df, diwali_df, post_diwali_df]

result_df = reduce(lambda left, right: left.merge(right, on="YEAR"), list_of_dfs)

In [30]:
result_df

,YEAR,AVG_PRE_PP_DAYS_SALES,AVG_PP_DAYS_SALES,NAVRATARI_FIRST_DAY_SALES,AVG_NAVRATARI_SALES,NAVRATARI_LAST_DAY_SALES,DUSSEHRA_SALES,AVG_PRE_DIWALI_DAYS_SALES,DHANTERAS_SALES,DIWALI_SALES,AVG_POST_DIWALI_DAYS_SALE
0,2023,9552.133333,3721.875000,57230.0,32199.7,34530.0,54390.0,32225.611111,248794.0,85119.0,16409.133333
1,2024,7741.933333,3394.562500,71797.0,36434.4,38061.0,59639.0,36351.222222,293987.0,85507.0,19085.066667
2,2025,9295.200000,2810.600000,46996.0,31604.6,32365.0,33493.0,42761.444444,245243.0,100113.0,21435.700000
3,2026,14606.400000,2733.933333,64462.0,44915.3,59173.0,29502.0,40313.666667,242579.0,110043.0,25637.241379


In [31]:
import pandas as pd
import plotly.express as px

# 1. Load the dataset
df = result_df.copy()

# 2. Define your exact custom order for the x-axis
custom_day_order = result_df.columns.tolist()[1:]

# 3. Reshape the data from "wide" to "long" format
df_melted = df.melt(
    id_vars=['YEAR'], 
    var_name='Type of Day', 
    value_name='Sales'
)

# 4. Ensure 'YEAR' is treated as a string/category for the legend
df_melted['YEAR'] = df_melted['YEAR'].astype(str)

# 5. Create the Plotly line chart
fig = px.line(
    df_melted, 
    x='Type of Day', 
    y='Sales', 
    color='YEAR',          
    markers=True,          # Adds dots at each data point
    title='Sales Trend by Type of Day (2023-2026)',
    category_orders={'Type of Day': custom_day_order}
)

# 6. Display the interactive chart
fig.show()

In [39]:
import pandas as pd
import plotly.graph_objects as go

# Select the two years to compare
year_a, year_b = 2025, 2026

def comparison_between_two_years_festive_days(year_a,year_b):
    df = result_df.copy()
    custom_day_order = result_df.columns.tolist()[1:]

    df_melted = df.melt(id_vars=['YEAR'], var_name='Type of Day', value_name='Sales')
    df_melted['YEAR'] = df_melted['YEAR'].astype(int)

    # Filter to the two years
    dfa = df_melted[df_melted['YEAR'] == year_a].set_index('Type of Day')['Sales']
    dfb = df_melted[df_melted['YEAR'] == year_b].set_index('Type of Day')['Sales']

    # Reindex to custom order
    dfa = dfa.reindex(custom_day_order)
    dfb = dfb.reindex(custom_day_order)

    pct_diff = ((dfb - dfa) / dfa * 100).round(1)

    fig = go.Figure()
    fig.add_trace(go.Bar(x=custom_day_order, y=dfa.values, name=str(year_a)))
    fig.add_trace(go.Bar(x=custom_day_order, y=dfb.values, name=str(year_b)))

    # Add % difference annotations above the taller bar
    for day in custom_day_order:
        val = pct_diff[day]
        y_pos = max(dfa[day], dfb[day])
        sign = "+" if val > 0 else ""
        fig.add_annotation(
            x=day, y=y_pos,
            text=f"{sign}{val}%",
            showarrow=False,
            yshift=15,
            font=dict(size=10, color="green" if val > 0 else "red"),
        )

    fig.update_layout(
        barmode='group',
        title=f'Sales Comparison: {year_a} vs {year_b} (with % change) Series A',
        xaxis_title='Type of Day',
        yaxis_title='Sales',
        xaxis_tickangle=-45,
    )

    fig.write_html(f"YOY_Sales_Comparison_of_prominent_festive_days_{year_a}_vs_{year_b}.html")

    fig.show()

In [40]:
comparison_between_two_years_festive_days(2025,2026)

In [12]:
#By Model name
import pandas as pd
import plotly.graph_objects as go
# from snowflake.snowpark.context import get_active_session

# session = get_active_session()

# === Pick your two years here ===
year_a, year_b = 2025,2026

df = session.table("MOP_DATABASE.SOQ.DAILY_FORECAST_MODEL_YEAR_WISE").to_pandas()

segment_df = pd.read_csv(r"C:\Users\G0004878\Downloads\View_Data_2026-09-21-1915.csv",sep='\t')

merged_df = pd.merge(left=df,right=segment_df,on=["MODEL_NAME"])

# Filter to selected years and pivot
comp = df[df['YEAR'].isin([year_a, year_b])].pivot(
    index='MODEL_NAME', columns='YEAR', values='TOTAL_NET_SALES'
).fillna(0)

comp = comp.sort_values(by=year_b, ascending=False)
models = comp.index.tolist()

vals_a = comp[year_a].values
vals_b = comp[year_b].values

# % change (handle zero division)
pct_diff = []
for a, b in zip(vals_a, vals_b):
    if a == 0:
        pct_diff.append(None)
    else:
        pct_diff.append(round((b - a) / a * 100, 1))

fig = go.Figure()
fig.add_trace(go.Bar(x=models, y=vals_a, name=str(year_a)))
fig.add_trace(go.Bar(x=models, y=vals_b, name=str(year_b)))

# Annotate % difference above taller bar
for i, model in enumerate(models):
    if pct_diff[i] is None:
        label = "New"
    else:
        sign = "+" if pct_diff[i] > 0 else ""
        label = f"{sign}{pct_diff[i]}%"

    y_pos = max(vals_a[i], vals_b[i])
    color = "green" if (pct_diff[i] is not None and pct_diff[i] > 0) else "red"

    fig.add_annotation(
        x=model, y=y_pos,
        text=label,
        showarrow=False,
        yshift=15,
        font=dict(size=10, color=color),
    )

fig.update_layout(
    barmode='group',
    title=f'Model-wise Sales Comparison: {year_a} vs {year_b} : Series A',
    xaxis_title='Model',
    yaxis_title='Total Net Sales',
    xaxis_tickangle=-45,
    height=600,
)

fig.write_html(f"Model_wise_sales_comparison_{year_a}_vs_{year_b}.html")
fig.show()

In [11]:
import pandas as pd
import plotly.graph_objects as go
# from snowflake.snowpark.context import get_active_session

# session = get_active_session()

# === Pick your two years here ===
year_a, year_b = 2025, 2026

df = session.table("MOP_DATABASE.SOQ.DAILY_FORECAST_ZO_YEAR_WISE").to_pandas()

# segment_df = pd.read_csv(r"C:\Users\G0004878\Downloads\View_Data_2026-09-21-1915.csv",sep='\t')

# df = pd.merge(left=df,right=segment_df,on=["MODEL_NAME"])

# Filter and pivot
comp = df[df['YEAR'].isin([year_a, year_b])].pivot(
    index='ZONAL_OFFICE_NAME', columns='YEAR', values='TOTAL_NET_SALES'
).fillna(0)

comp = comp.sort_values(by=year_b, ascending=False)
zones = comp.index.tolist()

vals_a = comp[year_a].values
vals_b = comp[year_b].values

# % change
pct_diff = []
for a, b in zip(vals_a, vals_b):
    if a == 0:
        pct_diff.append(None)
    else:
        pct_diff.append(round((b - a) / a * 100, 1))

fig = go.Figure()
fig.add_trace(go.Bar(x=zones, y=vals_a, name=str(year_a)))
fig.add_trace(go.Bar(x=zones, y=vals_b, name=str(year_b)))

for i, zone in enumerate(zones):
    if pct_diff[i] is None:
        label = "New"
    else:
        sign = "+" if pct_diff[i] > 0 else ""
        label = f"{sign}{pct_diff[i]}%"

    y_pos = max(vals_a[i], vals_b[i])
    color = "green" if (pct_diff[i] is not None and pct_diff[i] > 0) else "red"

    fig.add_annotation(
        x=zone, y=y_pos,
        text=label,
        showarrow=False,
        yshift=15,
        font=dict(size=11, color=color),
    )

fig.update_layout(
    barmode='group',
    title=f'Zonal Office Sales Comparison: {year_a} vs {year_b}',
    xaxis_title='Zonal Office',
    yaxis_title='Total Net Sales',
    xaxis_tickangle=-45,
    height=600,
)

fig.write_html(f"Zonal_Office_Sales_comparison_{year_a}_vs_{year_b}.html")

fig.show()

In [14]:
import pandas as pd
import plotly.graph_objects as go

# === Pick your two years here ===
year_a, year_b = 2025, 2026

# Aggregate sales by SEGMENT and YEAR
seg_df = merged_df.groupby(['YEAR', 'SEGMENT'], as_index=False)['TOTAL_NET_SALES'].sum()

# Filter and pivot
comp = seg_df[seg_df['YEAR'].isin([year_a, year_b])].pivot(
    index='SEGMENT', columns='YEAR', values='TOTAL_NET_SALES'
).fillna(0)

comp = comp.sort_values(by=year_b, ascending=False)
segments = comp.index.tolist()

vals_a = comp[year_a].values
vals_b = comp[year_b].values

# % change
pct_diff = []
for a, b in zip(vals_a, vals_b):
    if a == 0:
        pct_diff.append(None)
    else:
        pct_diff.append(round((b - a) / a * 100, 1))

fig = go.Figure()
fig.add_trace(go.Bar(x=segments, y=vals_a, name=str(year_a)))
fig.add_trace(go.Bar(x=segments, y=vals_b, name=str(year_b)))

for i, seg in enumerate(segments):
    if pct_diff[i] is None:
        label = "New"
    else:
        sign = "+" if pct_diff[i] > 0 else ""
        label = f"{sign}{pct_diff[i]}%"

    y_pos = max(vals_a[i], vals_b[i])
    color = "green" if (pct_diff[i] is not None and pct_diff[i] > 0) else "red"

    fig.add_annotation(
        x=seg, y=y_pos,
        text=label,
        showarrow=False,
        yshift=15,
        font=dict(size=10, color=color),
    )

fig.update_layout(
    barmode='group',
    title=f'Segment-wise Sales Comparison: {year_a} vs {year_b}',
    xaxis_title='Segment',
    yaxis_title='Total Net Sales',
    xaxis_tickangle=-45,
    height=600,
)
fig.write_html(f"Segment_wise_sales_comparison_{year_a}_vs_{year_b}.html")
fig.show()

In [20]:
parquet_pred_sf = parquet_pred_sf.with_column("MODEL_NAME",F.split_part(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),F.lit('_'),F.lit(2)))

In [21]:
parquet_pred_sf.show()

-------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"     |"CAL_DATE"           |"PRED_MEAN"          |"PRED_Q45"  |"PRED_Q55"  |"PRED_Q70"  |"PRED_Q75"  |"MODEL_NAME"  |
-------------------------------------------------------------------------------------------------------------------------------------------------------
|11132_XTREME 125_DRUM_SELF_CAST_GREY  |2026-09-01 00:00:00  |0.25808194490677366  |0.0         |0.0         |0.0         |0.0         |XTREME 125    |
|11132_XTREME 125_DRUM_SELF_CAST_GREY  |2026-09-02 00:00:00  |0.30974622314148803  |0.0         |0.0         |0.0         |0.0         |XTREME 125    |
|11132_XTREME 125_DRUM_SELF_CAST_GREY  |2026-09-03 00:00:00  |0.30396100088436645  |0.0         |0.0         |0.0         |0.0         |XTREME 125    |
|11132_XTREME 125_DRUM_SELF_CAST_GREY  |2026-09-04 00:00:00  |0.4155987251742711   |0.0 

In [22]:
parquet_pred = parquet_pred_sf.to_pandas()

In [23]:
parquet_pred = pd.merge(left=parquet_pred,right=segment_df,on=["MODEL_NAME"])

In [24]:
parquet_pred.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,CAL_DATE,PRED_MEAN,PRED_Q45,PRED_Q55,PRED_Q70,PRED_Q75,MODEL_NAME,SEGMENT
0,11885_SPLENDOR+_DRUM_SELF_CAST_BLACK SILVER,2026-11-16,0.001895,0.0,0.0,0.0,0.0,SPLENDOR+,100 CC
1,11885_SPLENDOR+_DRUM_SELF_CAST_BLACK SILVER,2026-11-17,0.001973,0.0,0.0,0.0,0.0,SPLENDOR+,100 CC
2,11885_SPLENDOR+_DRUM_SELF_CAST_BLACK SILVER,2026-11-18,0.001905,0.0,0.0,0.0,0.0,SPLENDOR+,100 CC
3,11885_SPLENDOR+_DRUM_SELF_CAST_BLACK SILVER,2026-11-19,0.001905,0.0,0.0,0.0,0.0,SPLENDOR+,100 CC
4,11885_SPLENDOR+_DRUM_SELF_CAST_BLACK SILVER,2026-11-20,0.002003,0.0,0.0,0.0,0.0,SPLENDOR+,100 CC


In [26]:
parquet_pred.groupby("SEGMENT").agg(SEGMENTAL_SALES=("PRED_Q70","sum")).to_csv(r"Segment_wise_sales.csv")